In [ ]:
!pip install -q transformers accelerate sentencepiece pandas scikit-learn

import pandas as pd
import json
import torch

from pathlib import Path
from google.colab import files, drive
from transformers import AutoTokenizer, AutoModelForCausalLM

drive.mount("/content/drive")

# Upload xed_train.tsv and xed_dev.tsv
needed = ["xed_train.tsv", "xed_dev.tsv"]

if any(not Path("/content", name).exists() for name in needed):
    files.upload()

train_df = pd.read_csv("/content/xed_train.tsv", sep="\t")
df = pd.read_csv("/content/xed_dev.tsv", sep="\t")

train_df = train_df.reset_index(drop=True)
df = df.reset_index(drop=True)

print("Training examples:", len(train_df))
print("Development examples:", len(df))

if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU before continuing.")

print("GPU:", torch.cuda.get_device_name(0))

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
).to("cuda")

model.eval()

print("Qwen loaded on:", model.device)

Mounted at /content/drive


Saving xed_dev.tsv to xed_dev.tsv
Saving xed_train.tsv to xed_train.tsv
Training examples: 3349
Development examples: 954
GPU: Tesla T4


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded on: cuda:0


In [ ]:
EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

examples_path = Path(
    "/content/drive/MyDrive/qwen_five_shot_examples.tsv"
)

# Preserve the exact same examples across sessions
if examples_path.exists():
    five_examples = pd.read_csv(examples_path, sep="\t")
else:
    five_examples = train_df.sample(
        n=5,
        random_state=42
    ).sort_index()

    five_examples.to_csv(
        examples_path,
        sep="\t",
        index=False
    )

print("Five training examples selected:")
print(five_examples[["english_text", "es_text", "emotion_labels"]])


def create_five_shot_prompt(sentence, language):

    column = "english_text" if language == "english" else "es_text"

    prompt = """Identify the emotions in this sentence.

Possible emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Return only a JSON array of the emotions expressed in the sentence.
Do not explain your answer.

"""

    for _, example in five_examples.iterrows():

        example_sentence = example[column]
        example_labels = json.loads(example["emotion_labels"])

        prompt += (
            f"Sentence: {example_sentence}\n"
            f"Answer: {json.dumps(example_labels)}\n\n"
        )

    prompt += f"Sentence: {sentence}\nAnswer:"

    return prompt


def parse_prediction(output):
    try:
        labels = json.loads(output.strip())

        if not isinstance(labels, list):
            return [], True

        if not all(isinstance(x, str) for x in labels):
            return [], True

        labels = [x.strip().lower() for x in labels]

        if not all(x in EMOTIONS for x in labels):
            return [], True

        return list(dict.fromkeys(labels)), False

    except (json.JSONDecodeError, TypeError, AttributeError):
        return [], True


# Check the format of one five-shot prompt
print("\nExample five-shot prompt:\n")
print(create_five_shot_prompt(df.iloc[0]["es_text"], "spanish"))

Five training examples selected:
                                           english_text  \
1236                               Give me the lighter.   
1552             Have enough respect for me not to lie.   
1830                             Wow... you look great.   
2628  Come off a rumble, we'd beat the shit out of a...   
2975  This magnificent structure will be the seat of...   

                                                es_text  \
1236                                   Dame el mechero.   
1552        Ten el suficiente respeto para no mentirme.   
1830  ¿Estamos de nuevo en terapia, doctor? Solo era...   
2628  Salimos de una terrible pelea, en la que derro...   
2975  Este gran edificio será sede de la nueva dirig...   

                    emotion_labels  
1236                     ["anger"]  
1552                     ["anger"]  
1830  ["joy", "surprise", "trust"]  
2628                     ["anger"]  
2975       ["anticipation", "joy"]  

Example five-shot prompt:

Identif

In [ ]:
def run_five_shot(language, limit=None, batch_size=4):

    assert language in ["english", "spanish"]

    column = "english_text" if language == "english" else "es_text"

    data = df if limit is None else df.head(limit)

    suffix = "full_dev" if limit is None else f"pilot{limit}"

    save_path = Path(
        f"/content/drive/MyDrive/qwen_five_shot_{language}_{suffix}.csv"
    )

    if save_path.exists():

        saved_df = pd.read_csv(save_path)

        results = saved_df.to_dict("records")
        completed = set(saved_df["row_index"].astype(int))

    else:
        results = []
        completed = set()

    remaining = [
        i for i in data.index
        if i not in completed
    ]

    print("Experiment:", language, "five-shot")
    print("Already completed:", len(completed))
    print("Remaining:", len(remaining))

    def save_checkpoint():
        if results:
            pd.DataFrame(results).to_csv(
                save_path,
                index=False
            )

    try:
        for start in range(0, len(remaining), batch_size):

            batch_indices = remaining[start:start + batch_size]

            prompts = [
                create_five_shot_prompt(
                    data.loc[i, column],
                    language
                )
                for i in batch_indices
            ]

            chat_prompts = [
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize=False,
                    add_generation_prompt=True
                )
                for prompt in prompts
            ]

            inputs = tokenizer(
                chat_prompts,
                padding=True,
                return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )

            input_length = inputs["input_ids"].shape[1]

            raw_outputs = tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )

            for i, raw in zip(batch_indices, raw_outputs):

                raw = raw.strip()
                predicted, malformed = parse_prediction(raw)

                actual = json.loads(
                    data.loc[i, "emotion_labels"]
                )

                results.append({
                    "row_index": int(i),
                    "sentence": data.loc[i, column],
                    "actual": json.dumps(actual),
                    "predicted": json.dumps(predicted),
                    "raw_output": raw,
                    "malformed": malformed
                })

            # Save every five batches
            if (start // batch_size + 1) % 5 == 0:
                save_checkpoint()
                print(f"Saved {len(results)}/{len(data)}")

    finally:
        save_checkpoint()

    print("Completed:", len(results))
    print("Saved to:", save_path)

    return results

In [ ]:
spanish_five_shot = run_five_shot(
    language="spanish",
    limit=None,
    batch_size=4
)

Experiment: spanish five-shot
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954
Completed: 954
Saved to: /content/drive/MyDrive/qwen_five_shot_spanish_full_dev.csv


In [ ]:
english_five_shot = run_five_shot(
    language="english",
    limit=None,
    batch_size=4
)

Experiment: english five-shot
Already completed: 0
Remaining: 954
Saved 20/954
Saved 40/954
Saved 60/954
Saved 80/954
Saved 100/954
Saved 120/954
Saved 140/954
Saved 160/954
Saved 180/954
Saved 200/954
Saved 220/954
Saved 240/954
Saved 260/954
Saved 280/954
Saved 300/954
Saved 320/954
Saved 340/954
Saved 360/954
Saved 380/954
Saved 400/954
Saved 420/954
Saved 440/954
Saved 460/954
Saved 480/954
Saved 500/954
Saved 520/954
Saved 540/954
Saved 560/954
Saved 580/954
Saved 600/954
Saved 620/954
Saved 640/954
Saved 660/954
Saved 680/954
Saved 700/954
Saved 720/954
Saved 740/954
Saved 760/954
Saved 780/954
Saved 800/954
Saved 820/954
Saved 840/954
Saved 860/954
Saved 880/954
Saved 900/954
Saved 920/954
Saved 940/954
Completed: 954
Saved to: /content/drive/MyDrive/qwen_five_shot_english_full_dev.csv


In [ ]:
import pandas as pd
import json

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    jaccard_score, hamming_loss, accuracy_score,
    classification_report
)

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

def evaluate_five_shot(language):

    path = (
        f"/content/drive/MyDrive/"
        f"qwen_five_shot_{language}_full_dev.csv"
    )

    results = pd.read_csv(path)

    assert len(results) == 954
    assert results["row_index"].nunique() == 954

    actual = results["actual"].apply(json.loads).tolist()
    predicted = results["predicted"].apply(json.loads).tolist()

    mlb = MultiLabelBinarizer(classes=EMOTIONS)

    y_true = mlb.fit_transform(actual)
    y_pred = mlb.transform(predicted)

    malformed = (
        results["malformed"]
        .astype(str)
        .str.lower()
        .eq("true")
        .mean()
    )

    metrics = {
        "language": language,
        "method": "five-shot",
        "examples": len(results),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "jaccard": jaccard_score(y_true, y_pred, average="samples", zero_division=0),
        "precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "exact_match": accuracy_score(y_true, y_pred),
        "malformed_rate": malformed
    }

    print(f"\nQWEN FIVE-SHOT {language.upper()}")
    print("-" * 40)

    for name, value in metrics.items():
        print(f"{name}: {value}")

    print("\nPer-emotion results:")
    print(classification_report(
        y_true, y_pred,
        target_names=EMOTIONS,
        zero_division=0
    ))

    return metrics


spanish_metrics = evaluate_five_shot("spanish")
english_metrics = evaluate_five_shot("english")

summary = pd.DataFrame([spanish_metrics, english_metrics])

summary.to_csv(
    "/content/drive/MyDrive/qwen_five_shot_metrics.csv",
    index=False
)

print("\nFive-shot metrics saved!")


QWEN FIVE-SHOT SPANISH
----------------------------------------
language: spanish
method: five-shot
examples: 954
micro_f1: 0.2921646746347942
macro_f1: 0.2688011330014055
jaccard: 0.2502620545073375
precision: 0.36223929747530187
recall: 0.24480712166172106
hamming_loss: 0.2095125786163522
exact_match: 0.16876310272536688
malformed_rate: 0.04507337526205451

Per-emotion results:
              precision    recall  f1-score   support

       anger       0.59      0.17      0.27       248
anticipation       0.55      0.08      0.14       216
     disgust       0.21      0.49      0.29       119
        fear       0.38      0.30      0.34       143
         joy       0.43      0.51      0.47       162
     sadness       0.35      0.42      0.38       138
    surprise       0.43      0.02      0.04       156
       trust       0.49      0.15      0.23       166

   micro avg       0.36      0.24      0.29      1348
   macro avg       0.43      0.27      0.27      1348
weighted avg       0

In [ ]:
import pandas as pd
import json
from pathlib import Path

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, jaccard_score

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

base = Path("/content/drive/MyDrive")

experiments = {
    "Spanish zero-shot": "qwen_zero_shot_spanish_full_dev.csv",
    "English zero-shot": "qwen_zero_shot_english_full_dev.csv",
    "Spanish five-shot": "qwen_five_shot_spanish_full_dev.csv",
    "English five-shot": "qwen_five_shot_english_full_dev.csv"
}

comparison = []

for name, filename in experiments.items():

    path = base / filename

    if not path.exists():
        print("Missing:", filename)
        continue

    data = pd.read_csv(path)

    if len(data) != 954 or data["row_index"].nunique() != 954:
        print("Incomplete:", name, len(data), "/954")
        continue

    actual = data["actual"].apply(json.loads).tolist()
    predicted = data["predicted"].apply(json.loads).tolist()

    mlb = MultiLabelBinarizer(classes=EMOTIONS)

    y_true = mlb.fit_transform(actual)
    y_pred = mlb.transform(predicted)

    comparison.append({
        "Experiment": name,
        "Micro-F1": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Macro-F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Jaccard": jaccard_score(
            y_true, y_pred, average="samples", zero_division=0
        ),
        "Malformed Rate": (
            data["malformed"]
            .astype(str)
            .str.lower()
            .eq("true")
            .mean()
        )
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df.to_string(index=False))

comparison_df.to_csv(
    base / "qwen_zero_vs_five_shot_comparison.csv",
    index=False
)

print("\nComparison saved!")

       Experiment  Micro-F1  Macro-F1  Jaccard  Malformed Rate
Spanish zero-shot  0.317184  0.281860 0.268187        0.045073
English zero-shot  0.344650  0.309472 0.288417        0.134172
Spanish five-shot  0.292165  0.268801 0.250262        0.045073
English five-shot  0.351267  0.327208 0.313155        0.057652

Comparison saved!


In [ ]:
from google.colab import files
import pandas as pd

files.upload()  # Select xed_test.tsv

test_df = pd.read_csv("xed_test.tsv", sep="\t")
test_df = test_df.reset_index(drop=True)

print("Test examples:", len(test_df))
print(test_df.head(3))

Saving xed_test.tsv to xed_test.tsv
Test examples: 483
                             es_text labels                      alignment_id  \
0         Bueno, eso acabó con esto.   1, 4  en/1965/59037/5735869.xml.gz::21   
1     Te dije que no me dijeras así.      1  en/1965/59037/5735869.xml.gz::23   
2  ¿Ustedes están ciegos o algo así?      1  en/1965/59037/5735869.xml.gz::35   

                             english_text     emotion_labels  
0               Well, that broke that up.  ["anger", "fear"]  
1           Told you not to call me that.          ["anger"]  
2  What, are you guys blind or something?          ["anger"]  


In [ ]:
import json
import torch
from pathlib import Path

def run_qwen_test(language, shot_type, batch_size=4):

    assert language in ["english", "spanish"]
    assert shot_type in ["zero", "five"]

    column = "english_text" if language == "english" else "es_text"

    save_path = Path(
        f"/content/drive/MyDrive/"
        f"qwen_{shot_type}_shot_{language}_TEST.csv"
    )

    if save_path.exists():
        previous = pd.read_csv(save_path)
        results = previous.to_dict("records")
        completed = set(previous["row_index"].astype(int))
    else:
        results = []
        completed = set()

    remaining = [
        i for i in test_df.index
        if i not in completed
    ]

    print(f"\nQWEN {language.upper()} {shot_type.upper()}-SHOT TEST")
    print("Already completed:", len(completed))
    print("Remaining:", len(remaining))

    def save_checkpoint():
        if results:
            pd.DataFrame(results).to_csv(save_path, index=False)

    try:
        for start in range(0, len(remaining), batch_size):

            indices = remaining[start:start + batch_size]

            prompts = []

            for i in indices:

                sentence = test_df.loc[i, column]

                if shot_type == "zero":
                    prompt = create_zero_shot_prompt(sentence)
                else:
                    prompt = create_five_shot_prompt(
                        sentence, language
                    )

                chat_prompt = tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    tokenize=False,
                    add_generation_prompt=True
                )

                prompts.append(chat_prompt)

            inputs = tokenizer(
                prompts,
                padding=True,
                return_tensors="pt"
            ).to(model.device)

            with torch.inference_mode():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=60,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id
                )

            input_length = inputs["input_ids"].shape[1]

            raw_outputs = tokenizer.batch_decode(
                outputs[:, input_length:],
                skip_special_tokens=True
            )

            for i, raw in zip(indices, raw_outputs):

                raw = raw.strip()
                predicted, malformed = parse_prediction(raw)

                actual = json.loads(
                    test_df.loc[i, "emotion_labels"]
                )

                results.append({
                    "row_index": int(i),
                    "sentence": test_df.loc[i, column],
                    "actual": json.dumps(actual),
                    "predicted": json.dumps(predicted),
                    "raw_output": raw,
                    "malformed": malformed
                })

            if (start // batch_size + 1) % 5 == 0:
                save_checkpoint()
                print(f"Saved {len(results)}/{len(test_df)}")

    finally:
        save_checkpoint()

    print("Completed:", len(results))
    print("Saved to:", save_path)

    return results

In [ ]:
def create_zero_shot_prompt(sentence):
    return f"""Identify the emotions in this sentence.

Possible emotions: anger, anticipation, disgust, fear, joy, sadness, surprise, trust.

Return only a JSON array of the emotions expressed in the sentence.
Do not explain your answer.

Sentence: {sentence}
Answer:"""

In [ ]:
qwen_test_zero_en = run_qwen_test(
    "english", "zero", batch_size=4
)


QWEN ENGLISH ZERO-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/qwen_zero_shot_english_TEST.csv


In [ ]:
qwen_test_zero_es = run_qwen_test(
    "spanish", "zero", batch_size=4
)


QWEN SPANISH ZERO-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/qwen_zero_shot_spanish_TEST.csv


In [ ]:
qwen_test_five_en = run_qwen_test(
    "english", "five", batch_size=4
)


QWEN ENGLISH FIVE-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/qwen_five_shot_english_TEST.csv


In [ ]:
qwen_test_five_es = run_qwen_test(
    "spanish", "five", batch_size=4
)


QWEN SPANISH FIVE-SHOT TEST
Already completed: 0
Remaining: 483
Saved 20/483
Saved 40/483
Saved 60/483
Saved 80/483
Saved 100/483
Saved 120/483
Saved 140/483
Saved 160/483
Saved 180/483
Saved 200/483
Saved 220/483
Saved 240/483
Saved 260/483
Saved 280/483
Saved 300/483
Saved 320/483
Saved 340/483
Saved 360/483
Saved 380/483
Saved 400/483
Saved 420/483
Saved 440/483
Saved 460/483
Saved 480/483
Completed: 483
Saved to: /content/drive/MyDrive/qwen_five_shot_spanish_TEST.csv


In [ ]:
from pathlib import Path
import pandas as pd

base = Path("/content/drive/MyDrive")

files_to_check = {
    "English zero-shot": "qwen_zero_shot_english_TEST.csv",
    "Spanish zero-shot": "qwen_zero_shot_spanish_TEST.csv",
    "English five-shot": "qwen_five_shot_english_TEST.csv",
    "Spanish five-shot": "qwen_five_shot_spanish_TEST.csv"
}

for name, filename in files_to_check.items():
    path = base / filename

    if path.exists():
        data = pd.read_csv(path)
        completed = data["row_index"].nunique()
        print(f"{name}: {completed}/483")
    else:
        print(f"{name}: NOT STARTED")

English zero-shot: 483/483
Spanish zero-shot: 483/483
English five-shot: 483/483
Spanish five-shot: 483/483


In [ ]:
import json
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    jaccard_score,
    hamming_loss,
    accuracy_score,
    classification_report
)

EMOTIONS = [
    "anger", "anticipation", "disgust", "fear",
    "joy", "sadness", "surprise", "trust"
]

test_metrics = []

for name, filename in files_to_check.items():

    data = pd.read_csv(base / filename)

    assert len(data) == 483, f"Incomplete: {name}"
    assert data["row_index"].nunique() == 483

    actual = data["actual"].apply(json.loads).tolist()
    predicted = data["predicted"].apply(json.loads).tolist()

    mlb = MultiLabelBinarizer(classes=EMOTIONS)
    y_true = mlb.fit_transform(actual)
    y_pred = mlb.transform(predicted)

    metrics = {
        "Experiment": name,
        "Micro-F1": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Macro-F1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "Jaccard": jaccard_score(
            y_true, y_pred, average="samples", zero_division=0
        ),
        "Precision": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Recall": recall_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "Hamming Loss": hamming_loss(y_true, y_pred),
        "Exact Match": accuracy_score(y_true, y_pred),
        "Malformed Rate": (
            data["malformed"].astype(str).str.lower().eq("true").mean()
        )
    }

    test_metrics.append(metrics)

    print(f"\n{name.upper()}")
    print(classification_report(
        y_true,
        y_pred,
        target_names=EMOTIONS,
        zero_division=0
    ))

summary = pd.DataFrame(test_metrics)

print("\nFINAL QWEN TEST RESULTS")
print(summary.to_string(index=False))

summary.to_csv(
    base / "qwen_final_test_metrics.csv",
    index=False
)

print("\nFinal Qwen test metrics saved!")


ENGLISH ZERO-SHOT
              precision    recall  f1-score   support

       anger       0.72      0.37      0.49       125
anticipation       0.38      0.59      0.46       109
     disgust       0.21      0.21      0.21        63
        fear       0.36      0.07      0.12        70
         joy       0.41      0.41      0.41        82
     sadness       0.36      0.26      0.30        70
    surprise       0.38      0.04      0.07        81
       trust       0.50      0.13      0.21        83

   micro avg       0.41      0.28      0.34       683
   macro avg       0.41      0.26      0.28       683
weighted avg       0.44      0.28      0.31       683
 samples avg       0.36      0.31      0.31       683


SPANISH ZERO-SHOT
              precision    recall  f1-score   support

       anger       0.55      0.18      0.28       125
anticipation       0.39      0.57      0.46       109
     disgust       0.16      0.19      0.17        63
        fear       0.37      0.10      0